##🎯 What you will learn:

✅ Understand the concept of Spatial Join and how it differs from traditional data joins

✅ Prepare and filter point data (POIs) for spatial analysis

✅ Assign each point (schools) to its corresponding governorate

✅ Visualize and export spatial analysis results



In [ ]:
# Import Libraries
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import os

In [ ]:
# mount google drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Define output folder directory
output_folder = '/content/drive/MyDrive/2-GeoPandas Mini Course/3-Spatial Join /Output'

In [ ]:
# Datasets Paths
# administrative boundaries (ADM1) Shapefile
adm_path = '/content/drive/MyDrive/2-GeoPandas Mini Course/2-GeoPackage Data Handling, CRS & Cleaning/Data/egy_admin1.shp'

# Egypt OSM data GeoPackage
osm_path = '/content/drive/MyDrive/2-GeoPandas Mini Course/2-GeoPackage Data Handling, CRS & Cleaning/Data/egypt.gpkg'


###Load the datasets

In [ ]:
# Load ADM1 Data
adm = gpd.read_file(adm_path)
adm.head()

In [ ]:
# Inspect GeoPackage Layers
osm_layers = gpd.list_layers(osm_path)
osm_layers

In [ ]:
# Load POIS Point Data
pois_points = gpd.read_file(osm_path, layer='gis_osm_pois_free')
pois_points

In [ ]:
# Check all unique vlaues in fclass
pois_points.fclass.unique()

In [ ]:
# Select only schools from POIs
schools = pois_points[pois_points['fclass'] == 'school']
schools

In [ ]:
# Select relevant columns(adm)
adm_sub = adm[['adm1_name', 'geometry']].rename(columns={'adm1_name': 'governorate'}).copy()
adm_sub.head()

In [ ]:
# Select relevant columns (schools)
schools_sub = schools[['name' , 'geometry']].copy()   # use .copy() to Create an independent copy to avoid modifying the original dataframe
schools_sub


In [ ]:
# Check CRS
print(adm_sub.crs)
print(schools_sub.crs)

In [ ]:
# Align CRS
adm_sub = adm_sub.to_crs(epsg=4326)
schools_sub = schools_sub.to_crs(epsg=4326)

###Spatial Join

In [ ]:
# Spatial Join to assign each school to a governorate
schools_joined = gpd.sjoin(schools_sub, adm_sub, how='left', predicate='within')
schools_joined

In [ ]:
# Drop index_right column
schools_joined = schools_joined.drop(columns=['index_right'])
schools_joined

### Map Plot

In [ ]:
# Visualize schools distribution across governorates
fig, ax = plt.subplots(figsize=(8,10))

# Plot administrative boundaries
adm_sub.plot(ax=ax, color='lightgray', edgecolor='black')

# Plot schools
schools_joined.plot(ax=ax, color='red', markersize=5)

plt.title("Schools Distribution Across Egyptian Governorates")
plt.tight_layout()
plt.show()

In [ ]:
# Count number of schools per governorates
counts = (
    schools_joined
    .groupby('governorate')
    .size()
    .reset_index(name='count')
)

In [ ]:
# Get top five governorates
top5 = counts.sort_values(by = 'count' , ascending = False).head(5).reset_index(drop=True)
top5

###Save outputs

In [ ]:
# Save as CSV (tabular data without geometry)
schools_joined.drop(columns='geometry').to_csv(
    os.path.join(output_folder, "schools_by_governorate.csv"),
    index=False
)

In [ ]:
# Save as GeoPackage (better)
schools_joined.to_file(
    os.path.join(output_folder, "schools_by_governorate.gpkg"),
    layer="schools_joined",
    driver="GPKG"
)

In [ ]:
# Save as shapefile
schools_joined.to_file(os.path.join(output_folder, "schools_joined.shp"))